### Testing knn accuracy of TF-IDF models

In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import datasets
import src
from src.knn_accuracy import knn_accuracy
from mteb.evaluation.evaluators.utils import get_vocab

Problem: the biorxiv version used here is not the same that is used for BiorxivClusteringP2P in MTEB! all Biorxiv results must be repeated with revision="65b79d1d13f80053f67aca9498d9402c2d9f1f40"

In [2]:
# load datasets
data_batched = {
"arxiv": datasets.load_dataset("mteb/arxiv-clustering-p2p", revision="a122ad7f3f0291bf49cc6f4d32aa80929df69d5d")["test"],
"biorxiv": datasets.load_dataset("mteb/biorxiv-clustering-p2p", revision="f5dbc242e11dd8e24def4c4268607a49e02946dc")["test"],
"medrxiv": datasets.load_dataset("mteb/medrxiv-clustering-p2p", revision="e7a26af6f3ae46b30dde8737f02c07b1505bcc73")["test"],
"reddit": datasets.load_dataset("mteb/reddit-clustering-p2p", revision="385e3cb46b4cfa89021f56c4380204149d0efe33")["test"],
"stackexchange": datasets.load_dataset("mteb/stackexchange-clustering-p2p", revision="815ca46b2622cec33ccafc3735d572c266efdb44")["test"]
}

In [3]:
data_full = {}
for name, data in data_batched.items():
    if name == "biorxiv":
        labels = [split["labels"] for split in data]
        sentences = [split["sentences"] for split in data]
    else:    
        labels = [x for split in data for x in split["labels"]]
        sentences = [x for split in data for x in split["sentences"]]
    data_full[name] = {"sentences": sentences, "labels": labels}

dataframe
colums: models
rows: datasets (full and batched)

Models to test:
- tfidf_log 
- tfidf_svd_log
- tfidf_svd_log_piecewise

#### logarithmic TF-IDF

In [ ]:
model = src.tfidf_log.Tfidf()
scores = {}

# get knn acc for full data 
for name, data in data_full.items():
    print(name)
    if name in ["arxiv", "reddit"]:
        # arxiv and reddit are too big to perform clustering on full tfidf representations
        continue            
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

# get knn acc for batched data
for name, data in data_batched.items():
    if name in ["biorxiv", "reddit"]: # biorxiv is the only one without batches
        continue
    # get full data to compute unified vocabulary
    vocab = get_vocab(data_full[name]["sentences"])
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores


with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

    

#### logarithmic TF-IDF with svd reduction

In [ ]:
def knn_acc_svd(model, dataset = data_full, dataset_batched = data_batched):
    "This is for model versions that perform SVD reduction with SVD matrix for the entire data"
    scores = {}
    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)
        # compute unified vocabulary (only necessary because the svd models need a vocab input)
        vocab = get_vocab(data["sentences"])
        # compute svd components
        v = model.encode(sentences=data["sentences"], vocab = vocab)
        # compute svd reduced embeddings
        embeddings = model.encode(sentences=data["sentences"], vocab = vocab, V = v)
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        
        if name == "biorxiv": # biorxiv is the only one without batches
            continue
        # get full data to compute unified vocabulary
        vocab = get_vocab(dataset[name]["sentences"])
        # compute svd components
        v = model.encode(sentences=dataset[name]["sentences"], vocab = vocab)
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab, V = v)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
        scores[name + "_batchwise"] = batch_scores
    

    return scores


In [ ]:
model = src.tfidf_svd_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"../MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


model = src.tfidf_svd50_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"../MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


model = src.tfidf_svd200_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"../MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


model = src.tfidf_svd300_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"../MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4)


model = src.tfidf_svd500_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"../MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

In [ ]:
def knn_acc(model, dataset = data_full, dataset_batched = data_batched):
    """This is for model versions that don't perform SVD or that use the old SVD implementation 
    where the SVD matrix is computed for each batch"""
    scores = {}

    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)            
        embeddings = model.encode(sentences=data["sentences"])
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        print(name)
        if name in ["biorxiv"]: # biorxiv is the only one without batches
            continue
        # get full data to compute unified vocabulary
        vocab = get_vocab(data_full[name]["sentences"])
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]

        scores[name + "_batchwise"] = batch_scores
    
    return scores

#### TF-IDF with svd for each batch (but vocab for the entire set)

In [ ]:
model = src.tfidf_svd_log_old.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

#### TF-IDF with svd with V and vocab for the each batch

In [ ]:
model = src.tfidf_svd_log_old.Tfidf()
scores = {}

# get knn acc for full data 
for name, data in data_full.items():
    print(name)          
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

# get knn acc for batched data
for name, data in data_batched.items():
    print(name)
    if name in ["biorxiv"]: # biorxiv is the only one without batches
        continue
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"])
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores


with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}_novocab.json", "w") as fp:
    json.dump(scores , fp, indent = 4)

#### TF-IDF with random projections

In [ ]:
model = src.tfidf_rnd100_log.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

In [ ]:
model = src.tfidf_rnd768_log.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

In [ ]:
######## this cell is executed on the cluster

# tfidf without reduction
model = src.tfidf_log.Tfidf()
scores = {}

# get knn acc for full data 
for name, data in data_full.items():
    print(name)
    if name in ["arxiv", "reddit"]:
        # arxiv and reddit are too big to perform clustering on full tfidf representations
        continue            
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

# get knn acc for batched data
for name, data in data_batched.items():
    if name in ["biorxiv", "reddit"]: # biorxiv is the only one without batches
        continue
    # get full data to compute unified vocabulary
    vocab = get_vocab(data_full[name]["sentences"])
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores


with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 



### tfidf with svd 500
model = src.tfidf_svd500_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


### tfidf with random projections
model = src.tfidf_rnd100_log.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

model = src.tfidf_rnd768_log.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

#### BIORXIV rerun with the correct dataset version 
this is gonna be ugly

In [ ]:
def knn_acc_bio(model, dataset = data_full, dataset_batched = data_batched):
    """ modified to include BIORXIV
    This is for model versions that don't perform SVD or that use the old SVD implementation 
    where the SVD matrix is computed for each batch"""
    scores = {}

    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)            
        embeddings = model.encode(sentences=data["sentences"])
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        print(name)
        # get full data to compute unified vocabulary
        vocab = get_vocab(data_full[name]["sentences"])
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]

        scores[name + "_batchwise"] = batch_scores
    
    return scores

In [ ]:
def knn_acc_svd(model, dataset = data_full, dataset_batched = data_batched):
    """ modified to include BIORXIV
    This is for model versions that perform SVD reduction with SVD matrix for the entire data"""
    scores = {}
    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)
        # compute unified vocabulary (only necessary because the svd models need a vocab input)
        vocab = get_vocab(data["sentences"])
        # compute svd components
        v = model.encode(sentences=data["sentences"], vocab = vocab)
        # compute svd reduced embeddings
        embeddings = model.encode(sentences=data["sentences"], vocab = vocab, V = v)
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        # get full data to compute unified vocabulary
        vocab = get_vocab(dataset[name]["sentences"])
        # compute svd components
        v = model.encode(sentences=dataset[name]["sentences"], vocab = vocab)
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab, V = v)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
        scores[name + "_batchwise"] = batch_scores
    

    return scores

In [ ]:
# load biorxiv datasets
data_batched = {
"biorxiv": datasets.load_dataset("mteb/biorxiv-clustering-p2p", revision="f5dbc242e11dd8e24def4c4268607a49e02946dc")["test"],
}
data_full = {}
for name, data in data_batched.items():
    labels = [x for split in data for x in split["labels"]]
    sentences = [x for split in data for x in split["sentences"]]
    data_full[name] = {"sentences": sentences, "labels": labels}